[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/electrostriction/electrostriction.ipynb)

# An elastic lattice of induced dipoles

An electric field polarizes each particle, and the dipoles pull on one another while elastic bonds resist their motion. Moving the particles changes the field that each dipole feels. The dipoles equilibrate through a dense field map; differentiating their equilibrated energy gives the mechanical forces and stiffness that determine the shape.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline

import jax
import numpy as np

from numga import Algebra, stack
from numga.backend.jax import JaxContext, derivative
from numga.sparse import SparseExtensor
from examples.animation import save_figure, show_animation
from examples.electromagnetism.electrostriction import render
from examples.electromagnetism.electrostriction.core import hexagonal_indices, strain

jax.config.update("jax_enable_x64", True)

In [ ]:
# The particles and their dipoles lie in a plane. Their electric interaction is the
# three-dimensional dipole field restricted to that plane, with four pi times permittivity one.
ga = Algebra("x+y+")
mv = JaxContext(ga, np.float64).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Response = ga.gatype((Vector, Vector))
Gradient = ga.gatype((Scalar, Vector))
Curvature = ga.gatype((Scalar, Vector, Vector))

## 1. The elastic lattice

Nearest neighbours share an elastic bond. Weaker tethers to the reference sites support the lattice and resist rigid motion. Each bond's link is the difference of its two particles' positions, one sparse map from the particles to the bonds. In the figures that follow, dashed bonds mark the reference shape, and deformation is shown at its actual scale.

In [ ]:
RINGS = 2
SPACING = 1.0
POLARIZABILITY = 0.035 * SPACING**3
STIFFNESS = 1.0
TETHER = 0.15 * STIFFNESS
FIELD = 3.05
NEWTON_STEPS = 4
DIPOLE_SCALE = 4.0
FIELD_SCALE = 0.65 / FIELD

indices, bonds = hexagonal_indices(RINGS)                                 # [particles, axes], [bonds, ends]
# The two lattice directions meet at sixty degrees.
across = (mv.xy * (-np.pi / 6)).exp() >> mv.x
axes = stack((mv.x, across))                                             # [axes] Vector
rest = (SPACING * (axes * indices).sum(axis=-1)).field()                 # Vector[particles]
particles = rest.gatype.site_shape[0]
# Each bond's second particle less its first.
ends = (np.ones_like(bonds) * [-1, 1])[..., None]                        # [bonds, ends, 1]
bond_ends = SparseExtensor.from_columns(bonds, mv.scalar(ends), particles)
rest_lengths = (bond_ends * rest).norm()                                 # Scalar[bonds]
save_figure(render.draw_rest(rest, bonds), "electrostriction_lattice")

## 2. The dipoles at a given shape

Each dipole feels the applied field and the fields of all the other dipoles. A map for every pair of particles becomes one field map with `.field(0, 1)`: its output and input slots both range over the particles. Solving it finds all the dipoles together. The orange arrows are the induced dipoles, and the gold arrow below the lattice is the applied field.

In [ ]:
applied = mv.x * FIELD


def response(positions: Vector) -> Response:
    sites = positions.batch()
    separation = sites[..., :, None] - sites[..., None, :]
    squared_distance = separation.squared()
    # Self pairs have a zero numerator, so the value added on the diagonal is arbitrary; it
    # keeps their denominator finite.
    distance = (squared_distance + np.eye(particles)).square_root()
    coupling = ((3 * separation * (separation | Vector) - squared_distance * Vector)
                / distance**5).field(0, 1)                              # Vector[particles] <- Vector[particles]
    return Vector / POLARIZABILITY - coupling


dipoles = response(rest).solve(applied)                                  # Vector[particles]
save_figure(render.draw_lattice(rest, bonds, rest, dipoles, applied,
                               DIPOLE_SCALE, FIELD_SCALE), "electrostriction_fixed")

## 3. Letting the lattice deform

At electrical equilibrium the polarization energy, including the cost of inducing the dipoles and the work of the fixed-field source, is minus half the dipoles' pairing with the applied field. The energy's first derivative is a form on a displacement field, and its second derivative pairs two such fields. Solving the second against the first gives a Newton displacement of the whole lattice.

In [ ]:
def total_energy(positions: Vector, field: Vector) -> Scalar:
    links = bond_ends * positions                                        # Vector[bonds]
    extension = links.norm() - rest_lengths
    elastic = STIFFNESS / 2 * extension.squared().sites.sum()
    supports = TETHER / 2 * (positions - rest).scalar_norm_squared().sites.sum()
    # The solve remains inside the differentiated function. Motion changes the interaction
    # between dipoles, and the dipoles readjust at every shape.
    polarization = response(positions).solve(field)
    electrical = -(field | polarization).sites.sum() / 2
    return elastic + supports + electrical


def equilibrium(field: Vector, steps: int) -> Vector:
    def energy(positions: Vector) -> Scalar:
        return total_energy(positions, field)

    positions = rest.broadcast_to(field.shape)
    for _ in range(steps):
        gradient: Gradient = derivative(energy)(positions)               # Scalar <- Vector[particles]
        curvature: Curvature = derivative(derivative(energy))(positions) # Scalar <- (Vector[particles], Vector[particles])
        positions = positions - curvature.solve(gradient)
    return positions


relaxed = equilibrium(applied, NEWTON_STEPS)
dipoles = response(relaxed).solve(applied)
save_figure(render.draw_lattice(rest, bonds, relaxed, dipoles, applied,
                               DIPOLE_SCALE, FIELD_SCALE), "electrostriction_relaxed")

## 4. Increasing the field

Reversing the applied field reverses every dipole and leaves the energy and shape unchanged. At weak fields the deformation therefore starts quadratically. The root mean square extent along each direction measures the lattice's changing shape as the field increases.

In [ ]:
LOAD_SAMPLES = 25
strengths = np.linspace(0, FIELD, LOAD_SAMPLES)
loads = mv.x * strengths                                                 # [loads] Vector
# The batch holds independent loading cases; the sites inside each field remain coupled.
loaded = equilibrium(loads, NEWTON_STEPS)                                 # [loads] Vector[particles]
parallel = strain(loaded, rest, mv.x)
transverse = strain(loaded, rest, mv.y)
save_figure(render.draw_loading(strengths, parallel, transverse), "electrostriction_loading")

## 5. Turning the applied field

The field rises smoothly, turns once and falls back to zero. Each frame is a mechanical equilibrium with its dipoles equilibrated as well; this is a slow loading cycle, with inertia and dielectric relaxation omitted. The orange arrows are the induced dipoles, and the gold arrow shows the applied field.

In [ ]:
FRAMES = 72
DURATION_MS = 70
phase = np.linspace(0, 2 * np.pi, FRAMES, endpoint=False)
strength = FIELD * (1 - np.cos(phase)) / 2
fields = ((mv.xy * (-phase / 2)).exp() >> mv.x) * strength                # [frames] Vector
shapes = equilibrium(fields, NEWTON_STEPS)                               # [frames] Vector[particles]
polarizations = response(shapes).solve(fields)
frames = render.animate_lattice(rest, bonds, shapes, polarizations, fields,
                                DIPOLE_SCALE, FIELD_SCALE)
show_animation(frames, "electrostriction", DURATION_MS)

The point dipoles have linear polarizability, and the elastic bonds and supporting tethers define a small model material. The sampled equilibria retain positive electrical and mechanical stiffness. The variational induced-dipole energy and its positional derivatives are discussed in [Simmonett et al., *Efficient treatment of induced dipoles*](https://pmc.ncbi.nlm.nih.gov/articles/PMC4545078/); the mechanical lattice here is a separate model of elastic support.